In [1]:
import pandas as pd
import numpy as np

In [24]:
parking = pd.read_csv("경기도 화성시_주정차단속현황_20250430.csv", encoding="cp949")
pop = pd.read_csv("유동인구_시군구_시간대별_집계.csv", encoding="cp949")
shop = pd.read_csv("소상공인시장진흥공단_상가(상권)정보_경기_202603.csv", encoding="utf-8")

In [25]:
# 주정차 단속 데이터 컬럼 확인
parking.columns

Index(['단속기관', '과태료 부과연월', '과태료 부과건수', '부과금액(천원)'], dtype='object')

In [26]:
parking = parking.rename(columns={
    "과태료 부과연월": "기준년월",
    "과태료 부과건수": "단속건수",
    "부과금액(천원)": "부과금액_천원"
})

parking["기준년월"] = pd.to_datetime(parking["기준년월"], errors="coerce")

parking["연도"] = parking["기준년월"].dt.year
parking["월"] = parking["기준년월"].dt.month  # 연도, 월 추출

parking["단속기관"] = parking["단속기관"].astype(str) # 문자열 처리

parking = parking.dropna(subset=["기준년월", "단속건수"]) # 결측치 제거

parking_monthly = parking.groupby(["연도", "월"], as_index=False).agg({
    "단속건수": "sum",
    "부과금액_천원": "sum"
}) # 월별 집계

In [27]:
# 유동인구 데이터 컬럼 확인
pop.columns

Index(['기준년월', '시군구코드', '시간대코드', '유동인구수', '유동인구수비율', '전월대비증감값', '전월대비증감비율',
       '전년동월대비증감값', '전년동월대비증감비율'],
      dtype='object')

In [36]:
pop = pop.rename(columns={
    "시간대코드": "시간대",
    "유동인구수": "유동인구수"
})

pop["기준년월"] = pd.to_datetime(pop["기준년월"], errors="coerce")

pop["연도"] = pop["기준년월"].dt.year
pop["월"] = pop["기준년월"].dt.month


pop_hs = pop[pop["시군구코드"] == 41590].copy() # 화성시 시군구코드: 41590

pop_hs = pop_hs[pop_hs["시간대"] == "TOT"].copy()

pop_hs = pop_hs.dropna(subset=["기준년월", "시간대", "유동인구수"])

pop_monthly = pop_hs.groupby(["연도", "월"], as_index=False).agg({"유동인구수": "sum"})

In [37]:
# 상권 데이터 컬럼 확인
shop.columns

Index(['상가업소번호', '상호명', '지점명', '상권업종대분류코드', '상권업종대분류명', '상권업종중분류코드',
       '상권업종중분류명', '상권업종소분류코드', '상권업종소분류명', '표준산업분류코드', '표준산업분류명', '시도코드',
       '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드',
       '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지',
       '건물부번지', '건물관리번호', '건물명', '도로명주소', '구우편번호', '신우편번호', '동정보', '층정보',
       '호정보', '경도', '위도'],
      dtype='object')

In [38]:
shop_hs = shop[shop["시군구명"].astype(str).str.contains("화성시")].copy() 

shop_hs = shop_hs[
    [
        "상호명",
        "상권업종대분류명",
        "상권업종중분류명",
        "상권업종소분류명",
        "시군구명",
        "행정동명",
        "법정동명",
        "지번주소",
        "도로명주소",
        "경도",
        "위도"
    ]
]

shop_hs = shop_hs.dropna(subset=["경도", "위도"])
shop_hs = shop_hs.drop_duplicates() 

# 업종별 상가 수
shop_category = shop_hs.groupby("상권업종대분류명", as_index=False).agg({"상호명": "count"}).rename(columns={"상호명": "상가수"})

# 행정동별 상가 수
shop_dong = shop_hs.groupby("행정동명", as_index=False).agg({
    "상호명": "count"
}).rename(columns={"상호명": "상가수"})

# 전체 상가 수
total_shop_count = len(shop_hs)

In [39]:
# 월별 통합 데이터 생성
final_monthly = pd.merge(
    parking_monthly,
    pop_monthly,
    on=["연도", "월"],
    how="left"
)

final_monthly["총상가수"] = total_shop_count

final_monthly["단속건수"] = final_monthly["단속건수"].fillna(0)
final_monthly["유동인구수"] = final_monthly["유동인구수"].fillna(0)

In [40]:
# 지역별 위험도를 정량적으로 비교할 수 있도록 파생변수 생성
final_monthly["건당평균부과금액_천원"] = (final_monthly["부과금액_천원"] / final_monthly["단속건수"]).replace([np.inf, -np.inf], 0).fillna(0)

final_monthly["유동인구_대비_단속비율"] = (final_monthly["단속건수"] / final_monthly["유동인구수"]).replace([np.inf, -np.inf], 0).fillna(0)

In [41]:
# 저장
final_monthly.to_csv("월별통합데이터_전처리.csv", index=False, encoding="utf-8-sig")
shop_hs.to_csv("화성시상권데이터_전처리.csv", index=False, encoding="utf-8-sig")
shop_category.to_csv("업종별상권집계_전처리.csv", index=False, encoding="utf-8-sig")
shop_dong.to_csv("행정동별상권집계_전처리.csv", index=False, encoding="utf-8-sig")

In [42]:
print("월별 통합 데이터")
display(final_monthly.head())

print("화성시 상권 데이터")
display(shop_hs.head())

print("업종별 상권 집계")
display(shop_category.head())

print("행정동별 상권 집계")
display(shop_dong.head())

월별 통합 데이터


,연도,월,단속건수,부과금액_천원,유동인구수,총상가수,건당평균부과금액_천원,유동인구_대비_단속비율
0,2021,1,13830,556613,371038.16,52635,40.246782,0.037274
1,2021,2,16421,658155,381731.06,52635,40.080080,0.043017
2,2021,3,21809,887194,411648.94,52635,40.680178,0.052980
3,2021,4,19742,798416,422630.30,52635,40.442508,0.046712
4,2021,5,19438,803594,419977.50,52635,41.341393,0.046283


화성시 상권 데이터


,상호명,상권업종대분류명,상권업종중분류명,상권업종소분류명,시군구명,행정동명,법정동명,지번주소,도로명주소,경도,위도
66,디앤아트,과학·기술,기술 서비스,기타 엔지니어링 서비스업,화성시 동탄구,동탄3동,능동,경기도 화성시 동탄구 능동 1064-5,경기도 화성시 동탄구 동탄원천로 354-28,127.058722,37.218269
69,에이치알씨앤씨,과학·기술,본사·경영 컨설팅,경영 컨설팅업,화성시 동탄구,동탄5동,영천동,경기도 화성시 동탄구 영천동 846-1,경기도 화성시 동탄구 동탄대로 677-12,127.100157,37.214934
70,사이로건축사사무소,과학·기술,기술 서비스,건축 설계 및 관련 서비스업,화성시 동탄구,동탄5동,영천동,경기도 화성시 동탄구 영천동 823-6,경기도 화성시 동탄구 동탄첨단산업1로 27,127.089472,37.211023
82,엘이엔티,과학·기술,기술 서비스,기타 엔지니어링 서비스업,화성시 만세구,향남읍,향남읍,경기도 화성시 만세구 향남읍 동오리 224-3,경기도 화성시 만세구 향남읍 발안로464번길 14-8,126.957157,37.129711
90,낼에프에이,과학·기술,기술 서비스,기타 엔지니어링 서비스업,화성시 병점구,화산동,안녕동,경기도 화성시 병점구 안녕동 176-172,경기도 화성시 병점구 안녕남로50번길 16,126.987531,37.196216


업종별 상권 집계


,상권업종대분류명,상가수
0,과학·기술,5073
1,교육,5404
2,보건의료,1315
3,부동산,2639
4,소매,12429


행정동별 상권 집계


,행정동명,상가수
0,기배동,548
1,남양읍,3507
2,동탄1동,4709
3,동탄2동,1074
4,동탄3동,1355
